In [0]:
from pyspark.sql import functions as F

# Tabela 1: vendedores (a tabela "pequena", de cadastro)
vendedores = spark.createDataFrame(
    [
        (1, "Ana",   "Sudeste"),
        (2, "Bruno", "Sul"),
        (3, "Carla", "Nordeste"),
        (4, "Diego", "Centro-Oeste"),   # não tem nenhuma venda
    ],
    "vendedor_id INT, nome STRING, regiao STRING"
)

# Tabela 2: vendas (a tabela "grande", de movimento)
vendas = spark.createDataFrame(
    [
        (101, 1, "Notebook", 3200.0),   # Ana tem 2 vendas
        (102, 1, "Mouse",      80.0),
        (103, 2, "Monitor",   950.0),
        (104, 3, "Teclado",   220.0),
        (105, 9, "Webcam",    150.0),   # vendedor_id 9 não existe no cadastro
    ],
    "venda_id INT, vendedor_id INT, produto STRING, valor DOUBLE"
)

vendedores.show()
vendas.show()

+-----------+-----+------------+
|vendedor_id| nome|      regiao|
+-----------+-----+------------+
|          1|  Ana|     Sudeste|
|          2|Bruno|         Sul|
|          3|Carla|    Nordeste|
|          4|Diego|Centro-Oeste|
+-----------+-----+------------+

+--------+-----------+--------+------+
|venda_id|vendedor_id| produto| valor|
+--------+-----------+--------+------+
|     101|          1|Notebook|3200.0|
|     102|          1|   Mouse|  80.0|
|     103|          2| Monitor| 950.0|
|     104|          3| Teclado| 220.0|
|     105|          9|  Webcam| 150.0|
+--------+-----------+--------+------+



In [0]:
### Inner join
vendedores.join(vendas,vendedores.vendedor_id == vendas.vendedor_id,how="inner").show()

+-----------+-----+--------+--------+-----------+--------+------+
|vendedor_id| nome|  regiao|venda_id|vendedor_id| produto| valor|
+-----------+-----+--------+--------+-----------+--------+------+
|          1|  Ana| Sudeste|     101|          1|Notebook|3200.0|
|          1|  Ana| Sudeste|     102|          1|   Mouse|  80.0|
|          2|Bruno|     Sul|     103|          2| Monitor| 950.0|
|          3|Carla|Nordeste|     104|          3| Teclado| 220.0|
+-----------+-----+--------+--------+-----------+--------+------+



In [0]:
### Left Join

vendas.join(vendedores,on="vendedor_id",how="left").show()

+-----------+--------+--------+------+-----+--------+
|vendedor_id|venda_id| produto| valor| nome|  regiao|
+-----------+--------+--------+------+-----+--------+
|          1|     101|Notebook|3200.0|  Ana| Sudeste|
|          1|     102|   Mouse|  80.0|  Ana| Sudeste|
|          2|     103| Monitor| 950.0|Bruno|     Sul|
|          3|     104| Teclado| 220.0|Carla|Nordeste|
|          9|     105|  Webcam| 150.0| NULL|    NULL|
+-----------+--------+--------+------+-----+--------+



In [0]:
### Right join
vendas.join(vendedores,on="vendedor_id",how="right").show()

+-----------+--------+--------+------+-----+------------+
|vendedor_id|venda_id| produto| valor| nome|      regiao|
+-----------+--------+--------+------+-----+------------+
|          1|     102|   Mouse|  80.0|  Ana|     Sudeste|
|          1|     101|Notebook|3200.0|  Ana|     Sudeste|
|          2|     103| Monitor| 950.0|Bruno|         Sul|
|          3|     104| Teclado| 220.0|Carla|    Nordeste|
|          4|    NULL|    NULL|  NULL|Diego|Centro-Oeste|
+-----------+--------+--------+------+-----+------------+



In [0]:
### Full join
vendas.join(vendedores,on="vendedor_id",how="full").show()

+-----------+--------+--------+------+-----+------------+
|vendedor_id|venda_id| produto| valor| nome|      regiao|
+-----------+--------+--------+------+-----+------------+
|          1|     101|Notebook|3200.0|  Ana|     Sudeste|
|          1|     102|   Mouse|  80.0|  Ana|     Sudeste|
|          2|     103| Monitor| 950.0|Bruno|         Sul|
|          3|     104| Teclado| 220.0|Carla|    Nordeste|
|          9|     105|  Webcam| 150.0| NULL|        NULL|
|          4|    NULL|    NULL|  NULL|Diego|Centro-Oeste|
+-----------+--------+--------+------+-----+------------+



In [0]:
vendedores.hint("shuflle_hash").join(vendas, "vendedor_id").explain()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- == Initial Plan ==
   PhotonResultStage
   +- PhotonColumnarToRow
      +- PhotonProject [vendedor_id#10668, nome#10669, regiao#10670, venda_id#10686, produto#10688, valor#10689]
         +- PhotonBroadcastHashJoin [vendedor_id#10668], [vendedor_id#10687], Inner, BuildLeft, false, true, false
            :- PhotonShuffleExchangeSource false
            :  +- PhotonShuffleMapStage EXECUTOR_BROADCAST, [id=#9236]
            :     +- PhotonShuffleExchangeSink SinglePartition
            :        +- PhotonRowToColumnar
            :           +- LocalTableScan [vendedor_id#10668, nome#10669, regiao#10670]
            +- PhotonRowToColumnar
               +- LocalTableScan [venda_id#10686, vendedor_id#10687, produto#10688, valor#10689]


== Photon Explanation ==
The query is fully supported by Photon.


In [0]:
vendedores.hint("broadcast").join(vendas, "vendedor_id").explain()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- == Initial Plan ==
   PhotonResultStage
   +- PhotonColumnarToRow
      +- PhotonProject [vendedor_id#10668, nome#10669, regiao#10670, venda_id#10686, produto#10688, valor#10689]
         +- PhotonBroadcastHashJoin [vendedor_id#10668], [vendedor_id#10687], Inner, BuildLeft, false, true, false
            :- PhotonShuffleExchangeSource false
            :  +- PhotonShuffleMapStage EXECUTOR_BROADCAST, [id=#9291]
            :     +- PhotonShuffleExchangeSink SinglePartition
            :        +- PhotonRowToColumnar
            :           +- LocalTableScan [vendedor_id#10668, nome#10669, regiao#10670]
            +- PhotonRowToColumnar
               +- LocalTableScan [venda_id#10686, vendedor_id#10687, produto#10688, valor#10689]


== Photon Explanation ==
The query is fully supported by Photon.
